# DiMEx model pipeline (Colab)

Trains and evaluates the valence-arousal regressors.

**Stage 1** learns the valence-arousal scale on DimABSA (SemEval-2026 Task 3, English restaurant and laptop).  
**Stage 2** adapts to drug reviews on Druglib; the validation slice is cut from Druglib's train file only, and the official test split stays held out.  
**Evaluate** scores the Druglib test split and Drugs.com (star ratings for satisfaction, plus a hand-labeled sample for effectiveness and side effects).

Before starting: *Runtime → Change runtime type → GPU* (T4 works; A100 or L4 is faster and supports bf16).

Checkpoints and results are saved to Google Drive, so you can run one model per session and pick up later.
Finished steps are skipped when you rerun the notebook, so a rerun only does what's missing.

**The comparison (2 × 2):** RoBERTa (general) and BioBERT (biomedical), each trained with and without the SemEval step.
Sections 4–6 run every model in `MODELS_TO_RUN`, skipping whatever is already finished, so rerunning the notebook only does what's missing. Section 8 draws the charts from whatever has finished.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/DiMEx'
WORK = '/content/drive/MyDrive/dimex'          # checkpoints, results and data live here
os.makedirs(WORK, exist_ok=True)

if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/av-jones/DiMEx.git {REPO}
%cd {REPO}
!pip -q install -r requirements-train.txt

## 2. Data

Downloads everything it can, and saves it to `My Drive/dimex/data/` so later sessions skip the download:
- **DimABSA**: cloned from the task's GitHub repo.
- **Druglib**: downloaded from UCI. If UCI blocks the download (it has blocked Drugs.com before), upload the Druglib zip to `My Drive/dimex/data/` and rerun this cell.
- **Drugs.com**: from a public GitHub mirror of the original UCI files.
- **Hand labels** (optional until you make them): `My Drive/dimex/data/drugscom_handlabels.csv` with columns `text, effectiveness, side_effects` (1-5 each).

In [ ]:
import glob, os, subprocess, zipfile
DATA = f'{WORK}/data'
DRUGLIB = f'{DATA}/druglib'
DRUGSCOM = f'{DATA}/drugscom/drugsComTest_raw.tsv'
HANDLABELS = f'{DATA}/drugscom_handlabels.csv'
os.makedirs(DRUGLIB, exist_ok=True); os.makedirs(os.path.dirname(DRUGSCOM), exist_ok=True)

# DimABSA
if not os.path.exists('/content/DimABSA2026'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DimABSA/DimABSA2026.git', '/content/DimABSA2026'], check=True)
# English Track A training files with valence-arousal labels. The task repo copies the same
# 'alltasks' file into subtask_1/2/3, so subtask_1 has everything; 'categorical' has polarity only.
DIMABSA = sorted(glob.glob('/content/DimABSA2026/task-dataset/track_a/subtask_1/eng/*train*.jsonl'))
print('DimABSA files:', *DIMABSA, sep='\n  ')

# Druglib: use a zip already in Drive if there is one, otherwise try UCI
if not glob.glob(f'{DRUGLIB}/*rain*.tsv'):
    zips = glob.glob(f'{DATA}/*rug*ib*.zip')
    if not zips:
        subprocess.run(['wget', '-q', '-O', f'{DATA}/druglib.zip',
                        'https://archive.ics.uci.edu/static/public/461/drug+review+dataset+druglib+com.zip'])
        zips = [f'{DATA}/druglib.zip']
    try:
        with zipfile.ZipFile(zips[0]) as z:
            for name in z.namelist():
                if name.endswith('.tsv'):
                    open(f'{DRUGLIB}/{os.path.basename(name)}', 'wb').write(z.read(name))
    except zipfile.BadZipFile:
        print('Druglib download was blocked. Upload the Druglib zip from UCI to My Drive/dimex/data/ and rerun.')

# Drugs.com test split from the GitHub mirror
if not os.path.exists(DRUGSCOM):
    subprocess.run(['wget', '-q', '-O', DRUGSCOM, 'https://raw.githubusercontent.com/semnan-university-ai/'
                    'Drug-Review-Dataset/main/drugsComTest_raw.tsv'])

print('Druglib files:', sorted(os.listdir(DRUGLIB)) or 'NONE')
for p in [DRUGSCOM, HANDLABELS]:
    print('found  ' if os.path.exists(p) and os.path.getsize(p) > 0 else 'MISSING', p)

## 3. Smoke test (about 2 minutes)
A tiny run (200 examples, one epoch) of each model that hasn't been trained yet, to catch problems before a long run. Optional.

In [ ]:
SMOKE_MODELS = ['ModernBERT', 'BioRedditBERT']
from core.registry import MODELS
from training.model import load_model
for m in SMOKE_MODELS:
    print(f'===== smoke test: {m}')
    _, tok = load_model(MODELS[m]['hf_id'] if m in MODELS else m)
    print('Tokenizer:', tok.tokenize('Lexapro gave me Nausea'))
    !python -m training.run stage1 --model {m} --dimabsa {' '.join(DIMABSA)} \
        --out /content/smoke/{m}/stage1 --epochs 1 --limit 200
    !python -m training.run stage2 --init /content/smoke/{m}/stage1 --druglib {DRUGLIB} \
        --out /content/smoke/{m}/stage2 --epochs 1 --limit 200

## 4. Train each model with the SemEval step
Stage 1 learns valence and arousal on SemEval (DimABSA); stage 2 adapts to Druglib.  
`MODELS_TO_RUN` takes names from `core/registry.py`. About 30 minutes per model on an A100; finished models are skipped, so only the new ones train.
Steps with a saved checkpoint are skipped; set `RETRAIN = True` to redo them.

In [ ]:
MODELS_TO_RUN = ['RoBERTa', 'ModernBERT',      # general web text
                 'BioBERT', 'BioRedditBERT']   # biomedical papers; + patient posts on Reddit
SEED = 42
BATCH = 16          # lower to 8 (and set GRAD_ACCUM = 2) if you run out of GPU memory
GRAD_ACCUM = 1
RETRAIN = False

from training.charts import slug
RES = f'{WORK}/results'
def ck(model, semeval=True):
    return f"{WORK}/ckpts/{slug(model)}{'' if semeval else '_nostage1'}/seed{SEED}"
def done(path): return not RETRAIN and os.path.exists(f'{path}/config.json')

for MODEL in MODELS_TO_RUN:
    CK = ck(MODEL)
    print(f'===== {MODEL} + SemEval')
    if done(f'{CK}/stage1'):
        print(f'Stage 1 already trained: {CK}/stage1')
    else:
        !python -m training.run stage1 --model {MODEL} --dimabsa {' '.join(DIMABSA)} --out {CK}/stage1 \
            --epochs 4 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}
    if done(f'{CK}/stage2'):
        print(f'Stage 2 already trained: {CK}/stage2')
    else:
        !python -m training.run stage2 --init {CK}/stage1 --druglib {DRUGLIB} --out {CK}/stage2 \
            --epochs 6 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}

## 5. Evaluate them
Scores the Druglib test split and Drugs.com, and saves each review's prediction for the confusion matrices.
A few minutes per model; skipped if a run already has saved predictions.

In [ ]:
extra = ''
if os.path.exists(DRUGSCOM):   extra += f' --drugscom {DRUGSCOM}'
if os.path.exists(HANDLABELS): extra += f' --handlabels {HANDLABELS}'

def evaluate(model, semeval=True):
    name = f"{slug(model)}{'' if semeval else '_nostage1'}_seed{SEED}"
    ckpt = ck(model, semeval)
    if not RETRAIN and os.path.exists(f'{RES}/preds/{name}.csv'):
        print(f'Already evaluated: {RES}/{name}.csv'); return
    if not os.path.exists(f'{ckpt}/stage2/config.json'):
        print(f'Not trained yet, skipping: {ckpt}/stage2'); return
    print(f'===== evaluating {name}')
    !python -m training.run evaluate --ckpt {ckpt}/stage2 --druglib {DRUGLIB} {extra} \
        --out {RES}/{name}.csv --preds {RES}/preds/{name}.csv

for MODEL in MODELS_TO_RUN:
    evaluate(MODEL)

## 6. Each model without the SemEval step (ablation)
Trains on Druglib only, starting from the plain pretrained model, with the same settings as section 4, then evaluates.
Comparing sections 5 and 6 shows whether learning valence and arousal on SemEval first helps. About 20 minutes per model.

In [ ]:
for MODEL in MODELS_TO_RUN:
    CK_ABL = ck(MODEL, semeval=False)
    print(f'===== {MODEL}, no SemEval')
    if done(f'{CK_ABL}/stage2'):
        print(f'Already trained: {CK_ABL}/stage2')
    else:
        !python -m training.run stage2 --init {MODEL} --druglib {DRUGLIB} --out {CK_ABL}/stage2 \
            --epochs 6 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}
    evaluate(MODEL, semeval=False)

## 7. Results table
Every finished run so far (averaged over seeds if you ran more than one). `kappa_3class` uses Gräßer et al.'s three classes.

In [ ]:
from training import charts
res = charts.load_results(WORK)
res.to_csv(f'{RES}/all_results.csv', index=False)
display(charts.results_table(res))

## 8. Charts
Each chart includes only the runs that have finished, so these can be rerun as more runs come in.

**8a. Training curves.** Training error vs. validation error per epoch. When training error keeps falling but validation error stays flat, the model is overfitting; the circled epoch is the checkpoint that was kept.

In [ ]:
charts.plot_training_curves(WORK)

**8b. Headline: Druglib → Drugs.com satisfaction.** The 2 × 2 grid. Blank cells are runs that haven't happened yet.

In [ ]:
display(charts.show_transfer_grid(res))

**8c. Compared with Gräßer et al. (2018).** Cohen's kappa per aspect for every run. For accuracy instead, use `metric='acc_3class'`.

In [ ]:
charts.plot_comparison(res, metric='kappa_3class')

**8d. Does the SemEval step help?** For each model trained both ways: kappa with the SemEval step minus kappa without it. Bars above zero mean it helped.

In [ ]:
charts.plot_ablation(res)

**8e. Confusion matrices.** Actual vs. predicted for one run. Darker diagonal = more correct; mistakes one step off the diagonal are near misses.
Effectiveness and side effects use Druglib's five levels; satisfaction uses Gräßer et al.'s three classes.

In [ ]:
for f in sorted(glob.glob(f'{RES}/preds/*.csv')):
    model_name, semeval, seed = charts.parse_run(os.path.basename(f)[:-4])
    charts.plot_confusion(f, title=f'{charts.run_label(model_name, semeval)}, seed {seed}')

## 9. Push a model to the Hugging Face Hub
Pushes one trained model (with the SemEval step) as a **private** repo. Then set its `ckpt` in `core/registry.py` on GitHub, and the app uses the real model.

In [ ]:
PUSH_MODEL = 'RoBERTa'
from huggingface_hub import notebook_login
notebook_login()
HF_REPO = 'YOUR_HF_USERNAME/dimex-' + slug(PUSH_MODEL)
!python -m training.run push --ckpt {ck(PUSH_MODEL)}/stage2 --repo {HF_REPO}